> 对应 Lec 5-8。覆盖矩阵计算四大原则、三种分解的 scipy 用法、分布式 OLS 完整流程、加权 OLS、流式读取大 CSV、岭回归 + 共轭梯度法、分布式 R² 计算。概念推导见"4. 线性回归_概念解释.md"。

---

## 矩阵计算四大原则代码示例

这四条原则是所有矩阵计算代码正误判断的依据。

### 原则一：先计算小维度（A^TAx 的正确计算顺序）

$A \in \mathbb{R}^{n\times p}$（$n \gg p$），计算 $A^TAx$：
- **错误顺序** $(A^TA)x$：先算 $A^TA$，复杂度 $O(np^2)$，结果是 $p\times p$ 矩阵
- **正确顺序** $A^T(Ax)$：先算 $Ax$（$O(np)$），再算 $A^T(\cdot)$（$O(np)$），总复杂度 $O(np)$，快 $p$ 倍

In [ ]:
import numpy as np

n, p = 100000, 500
A = np.random.randn(n, p)
x = np.random.randn(p)

# ❌ 错误：先算 p×p 矩阵，O(np²)
result_wrong = (A.T @ A) @ x

# ✅ 正确：先算 n×1 向量，O(np)
result_right = A.T @ (A @ x)    # 括号决定计算顺序！

# 两者结果相同，但速度差 p 倍

### 原则二：不要显式求逆（np.linalg.inv → np.linalg.solve）

求逆放大矩阵条件数，数值不稳定；且若矩阵稀疏，$A^{-1}$ 通常稠密，可能撑爆内存。"解方程"等价且更快更稳定。

In [ ]:
A = X.T @ X   # 对称正定矩阵
b = X.T @ y

# ❌ 错误：显式求逆
beta = np.linalg.inv(A) @ b

# ✅ 正确：转化为解线性方程组 Aβ = b
beta = np.linalg.solve(A, b)   # 内部选最优算法（LU/Cholesky），更稳定

### 原则三：对角矩阵广播代替显式构造（WX 不构造 n×n 的 W）

$W = \text{diag}(w)$ 是 $n\times n$ 的对角矩阵，当 $n=10^6$ 时 $W$ 需要 8TB 内存。用广播代替：

In [ ]:
w = np.random.uniform(1, 5, n)   # 对角元素向量，shape (n,)

# ❌ 错误：构造完整的 n×n 对角矩阵（内存灾难）
W = np.diag(w)                   # n=100万 时 8TB 内存
XtWX = X.T @ W @ X

# ✅ 正确：广播乘法代替矩阵乘法
# w[:, np.newaxis] 把 (n,) 变成 (n,1)，与 (n,p) 广播后每行乘对应权重
XtWX = X.T @ (X * w[:, np.newaxis])   # 等价于 X^T W X，但 O(np) 而非 O(n²p)
XtWy = X.T @ (w * y)                  # 等价于 X^T W y，w * y 逐元素乘

### 原则四：向量循环改为矩阵化操作（BLAS 加速）

NumPy 的矩阵运算底层调用 BLAS（基础线性代数子程序），使用 SIMD 指令并行执行，比 Python 循环快 100-1000 倍。

In [ ]:
# ❌ 错误：Python 循环逐行计算内积
scores = np.zeros(n)
for i in range(n):
    scores[i] = X[i] @ beta    # 每次一个内积，无法利用 BLAS

# ✅ 正确：整体矩阵-向量乘（BLAS GEMV，底层并行）
scores = X @ beta              # 一条语句，速度快 ~100×

---

## 三种矩阵分解的 scipy 用法

### LU 分解（lu_factor + lu_solve，一次分解多次求解）

**适用场景**：$A$ 是一般的非奇异方阵，没有对称正定等特殊结构。

**原理**：$A = PLU$，$P$ 是置换矩阵，$L$ 下三角，$U$ 上三角。分解一次 $O(n^3)$，之后每次求解 $O(n^2)$。若需要对同一个 $A$ 解多组右端向量（如多次迭代），先分解再多次求解可节省大量时间。

In [ ]:
from scipy.linalg import lu_factor, lu_solve

A = np.random.randn(500, 500)

# 一次分解（O(n³)），返回紧凑的 LU 分解结果
lu, piv = lu_factor(A)

# 多次求解（O(n²) 每次）
for b in [b1, b2, b3]:
    x = lu_solve((lu, piv), b)   # 不重新分解 A，直接用缓存的 LU

### Cholesky 分解（cho_factor + cho_solve，对称正定矩阵专用）

**适用场景**：$A$ 是**对称正定**矩阵，典型例子是 $X^TX$（$X$ 列满秩时）和 $X^TX + \lambda I$（岭回归、ADMM）。

**优势**：比 LU 快约 2 倍（利用对称性，只算下三角），且数值更稳定。

In [ ]:
from scipy.linalg import cho_factor, cho_solve

A = X.T @ X   # 对称正定
b = X.T @ y

# 分解（返回下三角因子和一个标志位 lower=True/False）
cho, low = cho_factor(A)

# 求解（使用已分解的因子，O(n²)）
beta = cho_solve((cho, low), b)

# 常见用法：ADMM 中 M = A^TA + ρI，分解一次，每轮迭代求解一次
rho = 1.0
M_cho = cho_factor(A.T @ A + rho * np.eye(p))   # 分解一次，O(p³)
for k in range(max_iters):
    rhs = Aty + rho * (z - u)
    x   = cho_solve(M_cho, rhs)    # 每轮 O(p²)，比重新分解快得多

### QR 分解（lstsq 或 linalg.qr + solve_triangular，数值最稳定）

**适用场景**：OLS 求解，特别是当 $X$ 存在接近多重共线性时。QR 分解直接操作 $X$，条件数为 $\kappa(X)$；而正规方程 $X^TX$ 的条件数是 $\kappa(X)^2$，QR 方式数值误差小得多。

**原理**：$X = Q_1 R_1$，代入正规方程化简得 $R_1\hat\beta = Q_1^Ty$，只需对上三角方程组做后向替换。

In [ ]:
from scipy.linalg import lstsq, qr, solve_triangular

# 方法一：直接调用 lstsq（推荐，内部用 QR，自动处理病态情况）
beta, residuals, rank, sv = lstsq(X, y)

# 方法二：手动 QR（便于理解底层过程）
Q1, R1 = np.linalg.qr(X, mode='reduced')   # 经济型 QR，Q1: (n,p), R1: (p,p)
# 求解上三角方程组 R1 @ beta = Q1.T @ y
beta = solve_triangular(R1, Q1.T @ y)       # 后向替换，O(p²)

### 三种分解选择逻辑（一般方阵/对称正定/病态数据）

| 情况 | 选择 | 原因 |
|------|------|------|
| $A$ 是一般非奇异方阵 | LU（`solve`） | 通用，稳定 |
| $A$ 是对称正定（如 $X^TX$） | Cholesky（`cho_factor/solve`） | 速度快 2× |
| $X$ 近似共线，追求精度 | QR（`lstsq`） | 条件数不平方，最稳定 |
| 同一个 $A$ 解多个 $b$ | LU 或 Cholesky 预分解 | 分解一次，多次 $O(n^2)$ 求解 |

---

## 分布式 OLS 实现模板（Map-Reduce + 可加性）

**算法思路**：OLS 正规方程 $X^TX\hat\beta = X^Ty$ 中，$X^TX = \sum_i X_i^TX_i$，$X^Ty = \sum_i X_i^Ty_i$。这种"可加性"是分布式 OLS 的理论基础：各节点只持有一块数据 $X_i$，本地计算局部贡献，主节点汇总后解方程。

### 可加性公式代码实现（X^TX = ΣX_i^TX_i，X^Ty = ΣX_i^Ty_i）

In [ ]:
import ray, numpy as np
from scipy.linalg import cho_factor, cho_solve

ray.init()

@ray.remote
def local_ols_stats(X_chunk, y_chunk):
    """计算一块数据的局部贡献"""
    XtX_local = X_chunk.T @ X_chunk   # (p, p)
    Xty_local = X_chunk.T @ y_chunk   # (p,)
    return XtX_local, Xty_local

### 含截距列的分布式 OLS 完整代码（hstack 全 1 列）

In [ ]:
n, p_raw = 200000, 50
X_raw = np.random.randn(n, p_raw)
beta_true = np.append([2.0], np.random.randn(p_raw))   # 真实系数（含截距）
y = np.hstack([np.ones((n, 1)), X_raw]) @ beta_true + np.random.randn(n) * 0.5

# 分块（每块约 20000 行）
num_blocks = 10
X_chunks = np.array_split(X_raw, num_blocks)
y_chunks = np.array_split(y, num_blocks)

tasks = []
for X_chunk, y_chunk in zip(X_chunks, y_chunks):
    # 每块加截距列
    X_block = np.hstack([np.ones((len(X_chunk), 1)), X_chunk])
    tasks.append(local_ols_stats.remote(ray.put(X_block), ray.put(y_chunk)))

results = ray.get(tasks)

XtX = sum(r[0] for r in results)    # 全局 X^TX
Xty = sum(r[1] for r in results)    # 全局 X^Ty

### Reduce 后用 Cholesky 求解（不用 solve 也不用 inv）

In [ ]:
# X^TX 是对称正定矩阵（X 列满秩），用 Cholesky 最优
cho, low = cho_factor(XtX)
beta_hat = cho_solve((cho, low), Xty)

print("截距估计:", beta_hat[0])
print("系数估计（前5）:", beta_hat[1:6])
print("真实值（前5）:", beta_true[1:6])

ray.shutdown()

---

## 流式读取大 CSV + 分布式计算

**算法思路**：当 CSV 文件超过内存限制时，`pd.read_csv(chunksize=N)` 返回迭代器，每次只读 N 行到内存。配合 `ray.put` 即读即发，内存中同时只有一个块，计算与读取可以流水线化。

### pd.read_csv(chunksize) 迭代器模式

In [ ]:
import pandas as pd

# chunksize 指定每次读取的行数，返回 TextFileReader 迭代器
for chunk in pd.read_csv("large_matrix.csv", header=None, chunksize=100000):
    # chunk 是一个 DataFrame，shape = (chunksize, ncols)
    mat = chunk.values    # 转为 numpy array
    # ... 处理 mat ...

### 每块 ray.put → remote 的边读边发写法

In [ ]:
ray.init()

@ray.remote
def compute_chunk_xtx(chunk_ref):
    mat = chunk_ref
    return mat.T @ mat

tasks = []
for chunk in pd.read_csv("large_matrix.csv", header=None, chunksize=100000):
    chunk_ref = ray.put(chunk.values)          # 推入共享内存
    tasks.append(compute_chunk_xtx.remote(chunk_ref))   # 立即派发，不等完成

results = ray.get(tasks)   # 读取完毕后统一等待
XtX = sum(results)

### skiprows 跳过注释行（pd.read_table 的用法）

考试数据文件常在开头有 `#` 注释行，`skiprows` 指定跳过的行数（从第0行开始计数）：

In [ ]:
# 文件格式：前2行是注释（# 开头），分隔符是逗号
# 第0列是 y，第1列起是特征 X
@ray.remote
def compute_xtx_xty(chunk_ref):
    chunk = chunk_ref           # (n, 1+p) 的 ndarray
    y_chunk = chunk[:, 0]
    X_chunk = np.hstack([np.ones((len(chunk), 1)), chunk[:, 1:]])  # 加截距
    return X_chunk.T @ X_chunk, X_chunk.T @ y_chunk

tasks = []
for chunk in pd.read_table(
    "sim_linear.txt",
    skiprows=2,           # 跳过前 2 行注释（skiprows=2 跳过索引 0,1 两行）
    delimiter=",",        # 分隔符（也可以是 ";" 等）
    header=None,          # 没有列名行
    chunksize=100000
):
    chunk_ref = ray.put(chunk.values)
    tasks.append(compute_xtx_xty.remote(chunk_ref))

results = ray.get(tasks)
XtX = sum(r[0] for r in results)
Xty = sum(r[1] for r in results)
beta_hat = np.linalg.solve(XtX, Xty)

---

## 加权 OLS 实现（X^TWX 高效计算）

**算法思路**：加权 OLS 的正规方程是 $X^TWX\hat\beta = X^TWy$，其中 $W = \text{diag}(w)$ 是权重对角矩阵。核心是用广播乘法替代显式构造 $n\times n$ 的 $W$（不可能的内存需求），将 $WX$ 转化为逐行乘权重。

### 广播代替对角矩阵：WX = X * w.reshape(-1, 1)

In [ ]:
# w 是权重向量，shape (n,)
w = np.random.uniform(1, 5, n)

# 关键等式：W @ X = X * w[:, np.newaxis]
# w[:, np.newaxis] 将 (n,) 变成 (n,1)，广播后每行乘对应权重
WX = X * w[:, np.newaxis]        # shape (n, p)，等价于 diag(w) @ X

# 因此：X^T W X = X^T (WX) = X^T @ (X * w[:, np.newaxis])
XtWX = X.T @ WX                  # 等价写法：X.T @ (X * w[:, np.newaxis])

# X^T W y = X^T (w * y)          # w * y 是逐元素相乘，shape (n,)
XtWy = X.T @ (w * y)

### 单机加权 OLS 完整代码（XtWX、XtWy、solve）

In [ ]:
import numpy as np
from scipy.linalg import cho_factor, cho_solve

n, p = 200000, 500
X = np.random.randn(n, p)
y = np.random.randn(n)
w = np.random.uniform(1, 5, n)

# 高效计算 X^TWX（不构造 W）
XtWX = X.T @ (X * w[:, np.newaxis])   # (p, p)，对称正定
XtWy = X.T @ (w * y)                  # (p,)

# Cholesky 求解（XtWX 是对称正定矩阵）
cho, low = cho_factor(XtWX)
beta_hat = cho_solve((cho, low), XtWy)

# 计算拟合值和残差
y_hat = X @ beta_hat
residuals = y - y_hat

### 分布式加权 OLS Map 函数（局部 XtWX_local + XtWy_local）

In [ ]:
@ray.remote
def weighted_ols_chunk(X_chunk, y_chunk, w_chunk):
    XtWX_local = X_chunk.T @ (X_chunk * w_chunk[:, np.newaxis])
    XtWy_local = X_chunk.T @ (w_chunk * y_chunk)
    return XtWX_local, XtWy_local

ray.init()

X_chunks = np.array_split(X, 10)
y_chunks = np.array_split(y, 10)
w_chunks = np.array_split(w, 10)

tasks = [
    weighted_ols_chunk.remote(ray.put(Xc), ray.put(yc), ray.put(wc))
    for Xc, yc, wc in zip(X_chunks, y_chunks, w_chunks)
]
results = ray.get(tasks)

XtWX_total = sum(r[0] for r in results)
XtWy_total = sum(r[1] for r in results)
beta_hat   = np.linalg.solve(XtWX_total, XtWy_total)

---

## 分布式 R² 计算

**算法思路**：$R^2 = 1 - SS_{res}/SS_{tot}$，其中 $SS_{res} = \sum(y_i - \hat y_i)^2$，$SS_{tot} = \sum(y_i - \bar y)^2$。两者都可以分块累加，但 $SS_{tot}$ 需要全局均值 $\bar y$，必须先单独计算 $\bar y$ 再广播。

### 分块计算局部 SS_res 和 SS_tot（含截距的预测值）

In [ ]:
@ray.remote
def compute_r2_chunk(X_chunk, y_chunk, beta, y_mean):
    y_hat = X_chunk @ beta
    ss_res = np.sum((y_chunk - y_hat) ** 2)
    ss_tot = np.sum((y_chunk - y_mean) ** 2)   # y_mean 是全局均值，主节点广播
    return ss_res, ss_tot

### 全局均值的获取（提前单独计算后广播）

In [ ]:
# 方法一：直接在主节点算（数据已在共享内存时）
y_mean = np.mean(y)

# 方法二：分布式计算全局均值（数据分散在各节点时）
@ray.remote
def local_sum_count(y_chunk):
    return y_chunk.sum(), len(y_chunk)

sum_tasks = [local_sum_count.remote(yc) for yc in y_chunks]
sum_results = ray.get(sum_tasks)
total_sum = sum(r[0] for r in sum_results)
total_n   = sum(r[1] for r in sum_results)
y_mean    = total_sum / total_n

### 汇总 R² = 1 - SS_res / SS_tot

In [ ]:
# 注意：X_chunks 需要包含截距列（若 beta 是含截距的回归系数）
tasks = [
    compute_r2_chunk.remote(ray.put(Xc), ray.put(yc), beta_hat, y_mean)
    for Xc, yc in zip(X_chunks, y_chunks)
]
results = ray.get(tasks)

ss_res_total = sum(r[0] for r in results)
ss_tot_total = sum(r[1] for r in results)
r_squared = 1.0 - ss_res_total / ss_tot_total
print(f"R² = {r_squared:.6f}")

---

## 岭回归闭式解实现

**算法思路**：岭回归在 OLS 的目标函数中加 $\lambda\|\beta\|^2$ 正则化，闭式解为 $\hat\beta = (X^TX + \lambda I)^{-1}X^Ty$。对角加 $\lambda I$ 后矩阵依然对称正定（严格正定，即使 $X$ 列不满秩），可以用 Cholesky 分解高效求解。

### (X^TX + λI)^{-1}X^Ty 的 Cholesky 实现（对角加 λ 后 cho_factor）

In [ ]:
from scipy.linalg import cho_factor, cho_solve

def ridge_regression(X, y, lam):
    """
    岭回归闭式解。
    lam: 正则化强度，lam > 0 保证矩阵正定（即使 X 列不满秩也能用 Cholesky）
    """
    p = X.shape[1]
    A = X.T @ X + lam * np.eye(p)   # 对角加 λI：使矩阵严格正定
    b = X.T @ y

    # Cholesky：A 此时一定对称正定
    cho, low = cho_factor(A)
    return cho_solve((cho, low), b)

# 示例
n, p = 10000, 500
X = np.random.randn(n, p)
y = X @ np.ones(p) + np.random.randn(n)

beta_ridge = ridge_regression(X, y, lam=1.0)

### λ 的取值策略（与数据尺度相关）

In [ ]:
# λ 的量纲与 X^TX 的量纲相同（约为 n × 特征量纲²）
# 实践中常用 λ/n 的形式（sklearn 默认 alpha=λ/n）
# 若 X 已标准化（列均值 0，方差 1），则 X^TX 对角元素约为 n，λ 在 n 量纲附近
lam_values = [0.01 * n, 0.1 * n, 1.0 * n]   # 对应 sklearn 的 alpha=0.01, 0.1, 1.0

---

## 共轭梯度法（CG）实现

**算法思路**：求解线性方程组 $Ax = b$（$A$ 对称正定）的迭代方法。CG 的关键优势是**无矩阵（matrix-free）**：不需要显式存储 $A$，只需要一个函数 `A_func(v)` 来计算矩阵-向量乘积 $Av$。对于岭回归，$Av = (X^TX + \lambda I)v = X^T(Xv) + \lambda v$，可以分解为两次 $O(np)$ 操作，完全不需要构造 $p\times p$ 的 $X^TX$。

### CG 算法核心代码（A_func 接口，不传矩阵）

In [ ]:
def conjugate_gradient(A_func, b, max_iter=None, tol=1e-8):
    """
    共轭梯度法求解 Ax = b，A 通过函数接口传入（无矩阵实现）。
    A_func: 接受向量 v，返回 A @ v
    b: 右端向量
    """
    p = len(b)
    if max_iter is None:
        max_iter = p          # CG 理论上最多 p 步精确收敛

    x = np.zeros(p)           # 初始猜测 x=0
    r = b.copy()              # 残差 r = b - A@x（初始 x=0 所以 r=b）
    d = r.copy()              # 搜索方向 d=r（初始时共轭方向等于残差）
    r_norm_sq = r @ r         # 残差的模平方，避免重复计算

    for _ in range(max_iter):
        if np.sqrt(r_norm_sq) < tol:
            break

        Ad = A_func(d)        # 核心：只需要矩阵-向量乘积，不需要矩阵本身

        # 步长 α = (r^T r) / (d^T A d)
        alpha = r_norm_sq / (d @ Ad)

        x = x + alpha * d     # 更新解
        r = r - alpha * Ad    # 更新残差

        r_norm_sq_new = r @ r
        # 方向更新系数 β = (r_new^T r_new) / (r_old^T r_old)
        beta = r_norm_sq_new / r_norm_sq
        d = r + beta * d      # 更新共轭方向

        r_norm_sq = r_norm_sq_new

    return x

### 无矩阵 MVP：(X^TX + λI)v = X^T(Xv) + λv 代码

In [ ]:
def ridge_matvec(X, v, lam):
    """
    计算 (X^TX + λI)v = X^T(Xv) + λv
    不构造 p×p 的矩阵，只需两次 O(np) 操作
    """
    return X.T @ (X @ v) + lam * v

# 使用示例
lam = 1.0
A_func = lambda v: ridge_matvec(X, v, lam)
b = X.T @ y

beta_cg = conjugate_gradient(A_func, b, tol=1e-8)

### 收敛判断（残差范数 < eps）与最大迭代次数

In [ ]:
# CG 理论保证：对 p×p 对称正定矩阵，最多 p 步精确收敛
# 实践中条件数好的矩阵往往几十步就收敛
# tol 设为 1e-8 到 1e-10，根据精度需求调整

# 验证：比较 CG 结果与 Cholesky 直接解
beta_cho = np.linalg.solve(X.T @ X + lam * np.eye(p), X.T @ y)
print("CG vs Cholesky 误差:", np.linalg.norm(beta_cg - beta_cho))

---

## 分布式岭回归（CG + Ray，一次缓存多次迭代）

**算法思路**：CG 的每一步都需要计算 $(X^TX+\lambda I)v = X^T(Xv)+\lambda v$，其中 $X^T(Xv)$ 可以分布式计算：各节点持有 $X_i$，并行计算 $X_i v$，主节点汇总得到 $Xv$，再分发给各节点计算 $X_i^T(Xv)$，汇总得到 $X^T(Xv)$。数据只在初始化时缓存一次，CG 的每次迭代只传小向量 $v$（$p$ 维）。

### 阶段一：流式读取 + ray.put 永久缓存 X 块 + 同时计算 b = X^Ty

In [ ]:
ray.init()

@ray.remote
def local_xty(chunk_ref):
    chunk = chunk_ref
    X_chunk = chunk[:, 1:]     # 第 0 列是 y
    y_chunk = chunk[:, 0]
    return X_chunk.T @ y_chunk, len(y_chunk)

X_chunk_ids = []   # 必须保留在变量中，防止 Ray GC 释放共享内存
Xty_tasks   = []

for chunk in pd.read_csv("data.csv", header=None, chunksize=100000):
    arr = chunk.values
    chunk_id = ray.put(arr)
    X_chunk_ids.append(chunk_id)    # 持有引用，数据不会被 GC
    Xty_tasks.append(local_xty.remote(chunk_id))

Xty_results = ray.get(Xty_tasks)
b = sum(r[0] for r in Xty_results)   # 全局 X^Ty

### 阶段二：定义 ray_ridge_mat_prod（每轮 CG 调用分布式 MVP）

In [ ]:
@ray.remote
def local_xv_xtvxv(chunk_ref, v):
    """计算局部 X_i^T(X_i v)，贡献给全局 X^T(Xv)"""
    chunk = chunk_ref
    X_chunk = chunk[:, 1:]
    Xiv = X_chunk @ v         # X_i v：局部矩阵-向量乘
    return X_chunk.T @ Xiv    # X_i^T (X_i v)

def distributed_ridge_matvec(v, X_chunk_ids, lam):
    """分布式计算 (X^TX + λI)v"""
    tasks = [local_xv_xtvxv.remote(cid, v) for cid in X_chunk_ids]
    partial = ray.get(tasks)
    XtXv = sum(partial)        # 汇总：Σ X_i^T(X_i v) = X^T(Xv)
    return XtXv + lam * v      # 加上正则项 λv

### 阶段三：调用 CG 得到 β̂

In [ ]:
lam = 1.0
p = b.shape[0]

A_func = lambda v: distributed_ridge_matvec(v, X_chunk_ids, lam)
beta_hat = conjugate_gradient(A_func, b, tol=1e-8)

print("岭回归系数（前5）:", beta_hat[:5])
ray.shutdown()

### X_chunk_ids 列表必须保留（防止 GC 释放 Ray 内存）

In [ ]:
# ❌ 错误：不保存 chunk_id，共享内存可能被 GC 释放
for chunk in pd.read_csv(...):
    ray.put(chunk.values)           # 返回的 ObjectRef 没有被任何变量持有
    tasks.append(local_xty.remote(...))   # 如果 GC 发生，数据已不在内存

# ✅ 正确：将所有 chunk_id 存入列表，列表存活则数据不会被 GC
X_chunk_ids = []
for chunk in pd.read_csv(...):
    chunk_id = ray.put(chunk.values)
    X_chunk_ids.append(chunk_id)   # 只要 X_chunk_ids 变量存在，数据就安全